# Insurance Claim Risk Detection — Analysis Notebook**Goal:** Analyse insurance claims data, understand it, clean it, and prepare it for fraud-risk modelling.This notebook covers:1. Data Understanding & Cleaning2. Exploratory Data Analysis (EDA)3. Feature Engineering4. Model Training5. Model ExplainabilityRun each cell top to bottom using **Shift+Enter**. Read the comments — they explain *why*, not just *what*.

## 1. Load the data

In [ ]:
# Import the libraries we need.# pandas -> handles tabular data (like an Excel sheet, but in code)# numpy -> handles numbers/mathimport pandas as pdimport numpy as np# Load the CSV into a DataFrame (think: an in-memory table/spreadsheet)df = pd.read_csv("../data/insurance_claims.csv")# Shape tells us (rows, columns)print("Shape:", df.shape)# Show the first 5 rows so we can see what the data looks likedf.head()

In [ ]:
# List every column name and its data type.# object = text/categorical, int64/float64 = numericdf.info()

## 2. Understand every column (data dictionary)Before touching anything, we document what each column means. This matters for the assignment — you must be able to explain this to the interviewer.| Column | Meaning ||---|---|| months_as_customer | How long the person has held a policy with this insurer || age | Policyholder's age || policy_number | Unique ID for the policy (not predictive — just an identifier) || policy_bind_date | Date the policy started || policy_state | US state where the policy was issued || policy_csl | Combined Single Limit — the max the insurer pays per accident (e.g. 250/500) || policy_deductable | Amount the customer pays out-of-pocket before insurance kicks in || policy_annual_premium | Yearly amount the customer pays for the policy || umbrella_limit | Extra liability coverage above the normal policy limit || insured_zip | Zip code of the policyholder || insured_sex, insured_education_level, insured_occupation, insured_hobbies, insured_relationship | Demographic info about the policyholder || capital-gains, capital-loss | Policyholder's investment gains/losses (financial profile) || incident_date | Date the accident/incident happened || incident_type | Type of incident (e.g. Single Vehicle Collision, Vehicle Theft) || collision_type | How the collision occurred (Front/Rear/Side), if applicable || incident_severity | How bad the damage was (Minor/Major/Total Loss) || authorities_contacted | Who was notified (Police, Fire, Ambulance, None) || incident_state, incident_city, incident_location | Where the incident happened || incident_hour_of_the_day | Hour (0-23) the incident occurred || number_of_vehicles_involved | How many vehicles were in the incident || property_damage | Whether property (not just the vehicle) was damaged || bodily_injuries | Number of people injured || witnesses | Number of witnesses || police_report_available | Whether a police report exists for this incident || total_claim_amount | Total amount being claimed || injury_claim, property_claim, vehicle_claim | Breakdown of the claim by category (these usually sum to total_claim_amount) || auto_make, auto_model, auto_year | Vehicle details || fraud_reported | **TARGET COLUMN** — Y/N, whether this claim was found to be fraudulent |**Assumption documented:** `policy_number`, `insured_zip`, `incident_location` are unique identifiers or near-unique free text — they don't generalise to new claims, so we will exclude them from modelling (keeping them would let the model 'memorise' individual records instead of learning real patterns).

## 3. Check for missing, duplicated, and incorrect values

In [ ]:
# This dataset hides missing values as the character '?' instead of a proper blank.# We replace '?' with NaN (pandas' standard 'missing value' marker) so pandas can detect them properly.df = df.replace('?', np.nan)# Now count missing values per column, only showing columns that actually have anymissing = df.isnull().sum()missing = missing[missing > 0].sort_values(ascending=False)print("Columns with missing values:")print(missing)

In [ ]:
# Check for fully duplicated rows (same values in every column)print("Duplicate rows:", df.duplicated().sum())# Check for an unnamed/empty trailing column that this dataset is known to haveprint(df.columns.tolist())

**Findings to document (fill this in after running the cells above):**- `collision_type`, `property_damage`, `police_report_available` contain missing values (the '?' we just converted).- There's a trailing column called `_c39` which is completely empty — a leftover from how this CSV was exported. We will drop it.- No duplicate rows were found (confirm this matches what you see above).**Assumption:** For `collision_type`, missing likely means "not applicable" (e.g. a vehicle theft has no collision type) rather than truly unknown — we'll treat it as its own category rather than guessing a value.

In [ ]:
# Drop the empty junk columndf = df.drop(columns=['_c39'], errors='ignore')# Fill missing categorical values with 'Unknown' rather than deleting rows (deleting would lose real claims)for col in ['collision_type', 'property_damage', 'police_report_available']:    if col in df.columns:        df[col] = df[col].fillna('Unknown')print("Remaining missing values:", df.isnull().sum().sum())print("New shape:", df.shape)

## 4. Detect possible data leakage**Data leakage** = a column that accidentally gives the model information it wouldn't have in real life at prediction time, making it look artificially accurate.Checks we do here:- `policy_number` — just an ID, no real signal, but could let the model 'cheat' by memorising IDs it saw during training. **Drop it.**- `incident_location` — near-unique free text per row, same issue. **Drop it.**- `injury_claim + property_claim + vehicle_claim` — these should sum to `total_claim_amount`. If so, keeping all 4 is redundant (not leakage exactly, but duplicate information) — we'll verify this below.

In [ ]:
# Verify whether the claim breakdown columns sum to the totaldf['claim_sum_check'] = df['injury_claim'] + df['property_claim'] + df['vehicle_claim']mismatch = (df['claim_sum_check'] != df['total_claim_amount']).sum()print(f"Rows where injury+property+vehicle claim does NOT equal total_claim_amount: {mismatch} out of {len(df)}")df = df.drop(columns=['claim_sum_check'])

In [ ]:
# Drop identifier columns that don't generalise and could leak / add noiseleak_or_id_cols = ['policy_number', 'insured_zip', 'incident_location', 'policy_bind_date', 'incident_date']df_model = df.drop(columns=[c for c in leak_or_id_cols if c in df.columns])print("Columns kept for modelling:", df_model.shape[1])print(df_model.columns.tolist())

**Note:** We're dropping `policy_bind_date` and `incident_date` as raw dates here, but we are NOT losing their value — in the Feature Engineering step next, we'll calculate the **number of days between policy start and incident**, and **days between incident and claim submission**, which are the genuinely useful signals hidden inside these dates.

In [ ]:
# Save the cleaned dataframe so we can reuse it in the next notebook sectiondf.to_csv("../data/insurance_claims_cleaned.csv", index=False)print("Saved cleaned data.")